# 4. Dashboard visuals

This notebook **prebuilds all dashboard visualization artifacts** (BupaR, DTW, FP-Growth) on **EC2** and **saves them to S3** for **direct dashboard integration**. The dashboard loads these prebuilt assets from S3 (the API returns only URLs; no computation at request time). Visuals are **SHAP/FFA-driven**: model data and feature lists come from Step 3b / 7 / 8 so process mining and itemset mining use only important features.

**Flow:** Run after [3_model_train_shap_ffa.ipynb](3_model_train_shap_ffa.ipynb). Then run [5_build_and_deploy.ipynb](5_build_and_deploy.ipynb) once to build and deploy.

## Steps

1. **Setup** – Resolve paths; create symlinks `10b_fpgrowth_dashboard_visual`, `10c_bupaR_dashboard_visual`, `10d_dtw_dashboard_visual` at repo root and under `9_risk_dashboard/visualizations` if missing.
2. **BupaR** – Process mining sequences and plots (SHAP/FFA allowed codes when available); **uploaded to the dashboard bucket** under `{S3_DASHBOARD_PREFIX}/bupar/{cohort}/{age_band}/plots/`.
3. **DTW** – Trajectory features and plots **based on SHAP/FFA important codes** (same as BupaR/FP-Growth); plot PNGs are **uploaded to the dashboard bucket** under `{S3_DASHBOARD_PREFIX}/dtw/{cohort}/{age_band}/plots/`. The DTW tab includes **appointments vs no appointments** visuals: **Routine vs No Routine (Outcomes)** and **High-Risk vs Low-Risk Trajectories** (outcome rate by trajectory intensity / archetype). These visuals use the **full pipeline (2016–2019)**: model_events (Step 4) and DTW features are built from all years 2016–2019, not a single year. **Extreme-density cohorts** (optional) support the same routine vs no routine comparison for high-utilizer subgroups—see optional step below.
4. **FP-Growth** – Itemsets, rules, **Plotly network HTML**, and PNGs; **uploaded to the dashboard bucket** (`S3_DASHBOARD_BUCKET`, e.g. jerome-dixon.io) under `{S3_DASHBOARD_PREFIX}/fpgrowth/{cohort}/{age_band}/plots/` (e.g. `vcu/pgx-risk-calculator/fpgrowth/...`). The dashboard loads the **network plot by cohort** from these URLs.
5. **Model performance metrics and cohort metadata** – Prebuilt via `generate_metrics.py` and `generate_metadata.py` (no recomputation). Deploy (5_build_and_deploy) uploads to the dashboard bucket: `metadata/model_performance_metrics.json` (Documentation tab) and `metadata/opioid_ed.json`, `metadata/non_opioid_ed.json` (dropdowns). Frontend loads these same-origin; Lambda GET /metrics and GET /metadata are fallbacks.
6. **API** – Returns URLs to prebuilt S3 assets only (no server-side computation for visuals).

Idempotent. Run from repo root. Prerequisites: notebook 5 done (`4_model_data`, `7_shap_analysis`, `8_ffa_analysis`); R and bupaR for BupaR.

In [ ]:
# Setup: paths and symlinks
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if (REPO_ROOT / "py_helpers").exists():
    pass  # already repo root
else:
    for p in REPO_ROOT.parents:
        if (p / "py_helpers").exists():
            REPO_ROOT = p
            break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

VISUAL_ROOT = REPO_ROOT / "9_risk_dashboard" / "visualizations"
BUPAR_SCRIPT = VISUAL_ROOT / "bupar" / "run_analysis.py"
DTW_FEATURES_SCRIPT = VISUAL_ROOT / "dtw" / "create_dtw_features.py"
DTW_ADD_SCRIPT = VISUAL_ROOT / "dtw" / "add_dtw_features_to_model_data.py"
FPGROWTH_SCRIPT = VISUAL_ROOT / "fpgrowth" / "run_analysis.py"

print(f"Repo root: {REPO_ROOT}")
print(f"Visualizations: {VISUAL_ROOT}")

In [ ]:
# Create symlinks 10b, 10c, 10d at repo root and under visualizations (idempotent: no-op if present)
def ensure_dashboard_symlinks():
    repo_links = [
        ("10c_bupaR_dashboard_visual", "9_risk_dashboard/visualizations/bupar"),
        ("10b_fpgrowth_dashboard_visual", "9_risk_dashboard/visualizations/fpgrowth"),
        ("10d_dtw_dashboard_visual", "9_risk_dashboard/visualizations/dtw"),
    ]
    for name, target in repo_links:
        path = REPO_ROOT / name
        target_path = REPO_ROOT / target
        if path.exists():
            print(f"  [repo] {name} exists")
            continue
        if not target_path.exists():
            continue
        try:
            path.symlink_to(target_path.relative_to(path.parent))
            print(f"  [repo] Created: {name}")
        except OSError as e:
            if os.name == "nt":
                print(f"  [repo] Windows: mklink /J \"{path}\" \"{target_path}\"")
            else:
                print(f"  [repo] {name}: {e}")
    for name, subdir in [("10c_bupaR_dashboard_visual", "bupar"), ("10b_fpgrowth_dashboard_visual", "fpgrowth"), ("10d_dtw_dashboard_visual", "dtw")]:
        path = VISUAL_ROOT / name
        if path.exists():
            continue
        target = VISUAL_ROOT / subdir
        if not target.exists():
            continue
        try:
            path.symlink_to(subdir)
            print(f"  [visual] Created: {name} -> {subdir}")
        except OSError as e:
            if os.name == "nt":
                print(f"  [visual] Windows: mklink /J \"{path}\" \"{target}\"")
            else:
                print(f"  [visual] {name}: {e}")

ensure_dashboard_symlinks()

## Config: cohorts and age bands

In [ ]:
from py_helpers.constants import COHORT_NAMES, AGE_BANDS

try:
    from py_helpers.constants import REQUIRED_COHORTS
except ImportError:
    REQUIRED_COHORTS = {"opioid_ed": ["13-24", "25-44", "45-54", "55-64"], "non_opioid_ed": ["65-74", "75-84", "85-94"]}

COHORTS_TO_RUN = []
AGE_BANDS_TO_RUN = []

# Default: only pipeline-supported (cohort, age_band) so model_events exist (no 0-12 for opioid_ed, etc.)
if not COHORTS_TO_RUN and not AGE_BANDS_TO_RUN:
    combinations = [(c, ab) for c, bands in REQUIRED_COHORTS.items() for ab in bands]
    print("Using pipeline-supported cohort/age_band (REQUIRED_COHORTS)")
else:
    if not COHORTS_TO_RUN:
        COHORTS_TO_RUN = COHORT_NAMES.copy()
    if not AGE_BANDS_TO_RUN:
        AGE_BANDS_TO_RUN = AGE_BANDS.copy()
    combinations = [(c, ab) for c in COHORTS_TO_RUN for ab in AGE_BANDS_TO_RUN]

print(f"Cohorts: {COHORTS_TO_RUN if COHORTS_TO_RUN else list(REQUIRED_COHORTS.keys())}")
print(f"Age bands: {AGE_BANDS_TO_RUN if AGE_BANDS_TO_RUN else 'per-cohort (REQUIRED_COHORTS)'}")
print(f"Total: {len(combinations)} combinations")

## Run BupaR process mining

Plots are uploaded to the dashboard bucket under `{S3_DASHBOARD_PREFIX}/bupar/{cohort}/{age_band}/plots/` (same pattern as FP-Growth).

In [ ]:
import subprocess

FAIL_FAST = True
for cohort_name, age_band in combinations:
    print(f"\n[BupaR] {cohort_name} / {age_band}")
    result = subprocess.run(
        [sys.executable, str(BUPAR_SCRIPT), "--cohort-name", cohort_name, "--age-band", age_band],
        cwd=str(REPO_ROOT),
    )
    if result.returncode != 0 and FAIL_FAST:
        raise RuntimeError(f"BupaR failed: {cohort_name} / {age_band}")
    print(f"  -> exit {result.returncode}")

## Run DTW trajectory features

**Trajectories are based on SHAP/FFA results** (same as BupaR/FP-Growth): `create_dtw_features.py` uses `get_shap_ffa_allowed_codes_combined()` to restrict trajectory events to SHAP/FFA important codes when available; if that is missing or empty, it falls back to all events in model_data. Run after Step 3b / 7 / 8 so SHAP/FFA outputs exist.

For each cohort/age band this step: (1) runs `create_dtw_features.py`, (2) runs `add_dtw_features_to_model_data.py` (merge + upload). **DTW plot PNGs** are uploaded to the **dashboard bucket** (same as FP-Growth/BupaR) under `{S3_DASHBOARD_PREFIX}/dtw/{cohort}/{age_band}/plots/` by `add_dtw_features_to_model_data.py` after the merge.

- **Expected filenames** (dashboard S3): `dtw_trajectory_analysis_{cohort}_{age_band}.png`, `dtw_sample_trajectories_{cohort}_{age_band}.png` (use underscore in age band, e.g. `0_12`). **chart_data.json** (routine_comparison, high_risk_trajectories) is also prebuilt and uploaded for direct dashboard integration.
- **Local paths** from which plots are uploaded (first existing wins):  
  `10d_dtw_dashboard_visual/outputs/{cohort}/{age_band_fname}/plots/` or  
  `5_feature_engineering/feature_engineering_outputs/6_dtw/{cohort}/{age_band}/plots/`.  
  Place PNGs in one of these so the add step uploads them; if no plots exist, upload is skipped.

The dashboard **DTW tab** shows **appointments vs no appointments**–related visuals: **Routine vs No Routine (Outcomes)** uses the **admin ICD filter** (`1b_apcd_event_filter/administrative_codes_lookup.json`): outcome rate for "No routine appointments (0 admin ICD events)" vs "Routine appointments (1+ admin ICD events)". **High-Risk vs Low-Risk Trajectories** shows outcome by trajectory archetype. DTW features include `admin_icd_event_count` (computed from model_events in create_dtw_features). **Date scope:** full pipeline (2016–2019).

In [ ]:
for cohort_name, age_band in combinations:
    print(f"\n[DTW] {cohort_name} / {age_band}")
    r1 = subprocess.run(
        [sys.executable, str(DTW_FEATURES_SCRIPT), "--cohort", cohort_name, "--age_band", age_band],
        cwd=str(REPO_ROOT),
    )
    if r1.returncode != 0 and FAIL_FAST:
        raise RuntimeError(f"DTW create_dtw_features failed: {cohort_name} / {age_band}")
    r2 = subprocess.run(
        [sys.executable, str(DTW_ADD_SCRIPT), "--cohort-name", cohort_name, "--age-band", age_band],
        cwd=str(REPO_ROOT),
    )
    if r2.returncode != 0 and FAIL_FAST:
        raise RuntimeError(f"DTW add_dtw_features failed: {cohort_name} / {age_band}")
    print(f"  -> DTW exit {r1.returncode}, {r2.returncode}")

## Appointments vs no appointments and extreme cohorts (optional)

**Research question (N1):** Is there a difference in outcomes for patients without routine appointments vs those with routine care? The DTW tab answers this via **Routine vs No Routine (Outcomes)** and **High-Risk vs Low-Risk Trajectories** (see above). These are shown over the **full pipeline (2016–2019)**, not a single year.

**Extreme-density cohorts** are high-utilizer patients (top ~5% by medical_code transaction density) split out so they do not dominate main models. They are for **exploratory visualization only** (see `docs/Step4_ModelData/README_model_data_and_extreme_split.md`). Running DTW (and optionally BupaR) for extreme cohorts lets you compare **routine vs no routine** in the high-utilizer subgroup. Run the cell below only if you have already run **Step 4** (model data) and want to build DTW (and optionally BupaR) visuals for an extreme-density cohort. Scripts: `extract_extreme_density_cohort.py`, then DTW/BupaR with cohort name `{source}_extreme_density` (e.g. `opioid_ed_extreme_density`).

In [ ]:
# Optional: run only if you want DTW (and BupaR) visuals for extreme-density cohorts.
# Leave EXTREME_COMBINATIONS empty to skip. Each pair is (source_cohort, age_band); script creates {cohort}_extreme_density.
EXTREME_COMBINATIONS = []  # e.g. [("opioid_ed", "25-44"), ("non_opioid_ed", "65-74")]

EXTREME_EXTRACT_SCRIPT = VISUAL_ROOT / "fpgrowth" / "extract_extreme_density_cohort.py"

for cohort_name, age_band in EXTREME_COMBINATIONS:
    print(f"\n[Extreme] Extract {cohort_name} / {age_band} -> {cohort_name}_extreme_density")
    r0 = subprocess.run(
        [sys.executable, str(EXTREME_EXTRACT_SCRIPT), "--cohort-name", cohort_name, "--age-band", age_band],
        cwd=str(REPO_ROOT),
    )
    if r0.returncode != 0 and FAIL_FAST:
        raise RuntimeError(f"Extract extreme cohort failed: {cohort_name} / {age_band}")
    extreme_name = f"{cohort_name}_extreme_density"
    print(f"[DTW] {extreme_name} / {age_band}")
    r1 = subprocess.run(
        [sys.executable, str(DTW_FEATURES_SCRIPT), "--cohort", extreme_name, "--age_band", age_band],
        cwd=str(REPO_ROOT),
    )
    if r1.returncode != 0 and FAIL_FAST:
        raise RuntimeError(f"DTW create_dtw_features failed: {extreme_name} / {age_band}")
    r2 = subprocess.run(
        [sys.executable, str(DTW_ADD_SCRIPT), "--cohort-name", extreme_name, "--age-band", age_band],
        cwd=str(REPO_ROOT),
    )
    if r2.returncode != 0 and FAIL_FAST:
        raise RuntimeError(f"DTW add_dtw_features failed: {extreme_name} / {age_band}")
    print(f"  -> DTW exit {r1.returncode}, {r2.returncode}")
    # Optional: run BupaR for extreme (if R script exists for this cohort)
    # subprocess.run([sys.executable, str(BUPAR_SCRIPT), "--cohort-name", extreme_name, "--age-band", age_band], cwd=str(REPO_ROOT))
if not EXTREME_COMBINATIONS:
    print("EXTREME_COMBINATIONS is empty; skipping extreme-density cohort extraction and DTW.")

## Run FP-Growth (itemsets, Plotly network HTML, S3 upload)

FP-Growth uses **SHAP/FFA-refined** model data: inputs come from `4_model_data` (built from Step 3b `cohort_feature_importance.csv`). For each cohort/age band this step: (1) ensures itemsets exist, (2) creates PNGs and **Plotly interactive network HTML**, (3) **uploads to the dashboard bucket** (e.g. `jerome-dixon.io`) under `{S3_DASHBOARD_PREFIX}/fpgrowth/{cohort}/{age_band}/plots/` (e.g. `vcu/pgx-risk-calculator/fpgrowth/...`). The dashboard then shows the **network plot for the user-selected cohort** via the `/visualizations/fpgrowth` API.

Run the cell below for each cohort/age band (builds itemsets, Plotly HTML, uploads to S3).

In [ ]:
for cohort_name, age_band in combinations:
    print(f"\n[FP-Growth] {cohort_name} / {age_band}")
    result = subprocess.run(
        [sys.executable, str(FPGROWTH_SCRIPT), "--cohort-name", cohort_name, "--age-band", age_band],
        cwd=str(REPO_ROOT),
    )
    if result.returncode != 0 and FAIL_FAST:
        raise RuntimeError(f"FP-Growth failed: {cohort_name} / {age_band}")
    print(f"  -> exit {result.returncode}")

## API (reference)

Lambda receives **user input** (cohort, age_band, model/feature selections) and **filters** only—it does not process or generate visualization data. All BupaR, DTW, and FP-Growth visuals are **prebuilt on EC2** and **saved to S3**; the API returns **URLs** to those prebuilt assets (filtered by cohort/age_band). Endpoints: `GET /visualizations/causal`, `/visualizations/bupar`, `/visualizations/dtw`, `/visualizations/fpgrowth`. See `9_risk_dashboard/backend/README.md`.

In [ ]:
print("Dashboard endpoints: 9_risk_dashboard/backend/README.md")
print("API Gateway deploy: utility_scripts/create_api_gateway_pgx_risk_calculator.sh")

## Next: Build and deploy

Build and deploy run **only** in [5_build_and_deploy.ipynb](5_build_and_deploy.ipynb). Run that notebook after this one.

In [ ]:
# Build and deploy run only in 5_build_and_deploy.ipynb. Run that notebook after this one.
print("Build and deploy (once): open 5_build_and_deploy.ipynb and run it after this notebook.")

*(Build and deploy — including frontend sync to S3 — are done only in notebook 3. See above.)*

In [ ]:
# No-op: build and deploy run only in 5_build_and_deploy.ipynb
pass